# Dataspace Contract Negotiation & PULL Transfer Flow

This notebook demonstrates an end-to-end Dataspace Protocol (DSP 2025-1) lifecycle in PULL mode:
1. Executes prerequisite setup (`setup_catalog_policies_oauth.ipynb`) to initialize participants, catalog, policies, and secrets.
2. Performs full DSP 2025-1 Contract Negotiation between Consumer and Provider (Request -> Offer -> Counter-Request -> Acceptance -> Agreement -> Verification -> Finalization).
3. Executes DSP PULL Transfer Process using Eunomia RPC (`/dsp/current/transfers/rpc/setup-*`).
4. Exercises transfer lifecycle state transitions: `REQUESTED` -> `STARTED` -> `SUSPENDED` -> `STARTED` (resumed).
5. Validates protocol enforcement rules (role constraint check and JSON-LD RDF shape validation).
6. Terminates the transfer process and verifies final state via the Transfer Agent Management API.

## 1. Prerequisites
Execute the catalog and policies notebook to establish onboarding, catalog entries, policies, and bearer token secrets.

In [36]:
%run ./setup_catalog_policies_oauth.ipynb

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500)
Wallet linked for Consumer (http://127.0.0.1:1100)
Wallet linked for Provider (http://127.0.0.1:1200)
Authority DID: did:web:127.0.0.1%3A1500

Consumer DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ

Provider DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dE

# Part I: Contract Negotiation (DSP 2025-1)
Complete negotiation sequence between Consumer and Provider following DSP 2025-1 state transitions.

## 2. Negotiation Request Initiation (Consumer -> Provider)
The Consumer initiates contract negotiation by sending a `ContractRequestMessage` via `/dsp/current/negotiations/rpc/setup-request-init` referencing the target dataset and offer policy.
The Provider assigns a `providerPid` and transitions the negotiation state to `REQUESTED`.

In [37]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:95b2f3d0-1b8c-417e-aabe-daf70bff1ee7",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
  

## 3. Provider Initial Offer (Provider -> Consumer)
The Provider responds with a counter-offer by sending a `ContractOfferMessage` via `/dsp/current/negotiations/rpc/setup-offer`.
The negotiation transitions to `OFFERED`.

In [21]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:7ed41a31-606d-453e-bb27-14460b59b896",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:4538386c-10f4-40e8-835a-4dbf47f068f7"
    },
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705",
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:7d4a436d-48d9-467f-a1

## 4. Consumer Counter-Request Based on Offer (Consumer -> Provider)
The Consumer reviews the offer and submits a counter-request via `/dsp/current/negotiations/rpc/setup-request`.
The negotiation state transitions back to `REQUESTED`.

In [22]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:7ed41a31-606d-453e-bb27-14460b59b896",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:4538386c-10f4-40e8-835a-4dbf47f068f7"
    },
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705",
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:8a247f66-e948-4dbe-b57c-1b5a61391124",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgen

## 5. Provider Offer Confirmation (Provider -> Consumer)
The Provider confirms or updates the terms of the offer via `/dsp/current/negotiations/rpc/setup-offer`.
The negotiation state transitions to `OFFERED`.

In [23]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:7ed41a31-606d-453e-bb27-14460b59b896",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:4538386c-10f4-40e8-835a-4dbf47f068f7"
    },
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705",
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:7d4a436d-48d9-467f-a1ab-722d9fdb6cd9",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "OFFERED",
    "stateAttribute": null,
    "associated

## 6. Consumer Offer Acceptance (Consumer -> Provider)
The Consumer accepts the Provider's offer terms via `/dsp/current/negotiations/rpc/setup-acceptance`.
The negotiation state transitions to `ACCEPTED`.

In [24]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705",
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:8a247f66-e948-4dbe-b57c-1b5a61391124",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdH

## 7. Provider Contract Agreement (Provider -> Consumer)
The Provider generates the binding `ContractAgreementMessage` containing the finalized agreement terms and sends it to the Consumer via `/dsp/current/negotiations/rpc/setup-agreement`.
The negotiation state transitions to `AGREED`.

In [25]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705",
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:7d4a436d-48d9-467f-a1ab-722d9fdb6cd9",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQ

## 8. Consumer Agreement Verification (Consumer -> Provider)
The Consumer verifies receipt and terms of the contract agreement by sending a `ContractAgreementVerificationMessage` via `/dsp/current/negotiations/rpc/setup-verification`.
The negotiation state transitions to `VERIFIED`.

In [26]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705",
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:8a247f66-e948-4dbe-b57c-1b5a61391124",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdH

## 9. Provider Negotiation Finalization (Provider -> Consumer)
The Provider acknowledges verification and sends a `ContractNegotiationEventMessage` (finalization) via `/dsp/current/negotiations/rpc/setup-finalization`.
The negotiation transitions to the terminal state `FINALIZED`.

In [27]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5af7258e-a69b-4da2-aa80-6e882dc78705",
    "providerPid": "urn:provider-pid:b787d164-43f2-45e5-9d1c-72a8ed390604",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:7d4a436d-48d9-467f-a1ab-722d9fdb6cd9",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd

## 10. Final Agreement Details
Display the finalized contract agreement and extract the `agreement_id` required for subsequent transfer requests.

In [28]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:3940c7db-118d-434b-9017-7e01de13f168",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Public",
  "negotiationAgentProcessId": "urn:negotiation-process:7d4a436d-48d9-467f-a1ab-722d9fdb6cd9",
  "negotiationAgentMessageId": "urn:negotiation-message:c445b305-916c-4176-a97f-f59fc1e038e9",
  "consumerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "providerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXN

# Part II: Transfer Process Execution (PULL Mode via Eunomia RPC)

The Transfer Agent exposes outbound RPC endpoints (`/dsp/current/transfers/rpc/setup-*`) that accept JSON-LD messages typed in the Eunomia context (`https://w3id.org/eunomia/v1/context.jsonld`).
The agent translates them into canonical DSP 2025-1 transfer protocol messages and exchanges them over the wire with the peer's DSP endpoint.

In **PULL mode**, the Provider provides a data address (endpoint and authorization token) from which the Consumer directly pulls data.

## 11. Transfer Request Setup (Consumer -> Provider)
The Consumer initiates the PULL transfer by sending a `RpcTransferRequestMessage` via `/dsp/current/transfers/rpc/setup-request` referencing the contract agreement ID.
The Provider creates a transfer process in `REQUESTED` state and returns its `providerPid`.

In [29]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["transferAgentModel"]["correlation"]["consumerPid"]
transfer_process_provider_pid = response_as_json["transferAgentModel"]["correlation"]["providerPid"]
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "agreementId": "urn:agreement:3940c7db-118d-434b-9017-7e01de13f168",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "format": "http+asd",
    "providerAddress": "http://127.0.0.1:1200/dsp/current"
  },
  "response": {
    "@conte

## 12. Start Transfer (Provider -> Consumer)
According to DSP 2025-1, in a PULL transfer the Provider sends the `TransferStartMessage` to the Consumer, transmitting the data endpoint from which the Consumer can pull data.
The Provider invokes `/dsp/current/transfers/rpc/setup-start`, transitioning the transfer state from `REQUESTED` to `STARTED`.

In [30]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
    "providerPid": "urn:uuid:3b969c05-293f-4a82-9124-b2702db02fe9"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:ef4638c5-d0df-4a9f-bb5f-f5e09163dba6",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
      "providerPid": "urn:uuid:3b969c05-293f-4a82-9124-b2702db02fe9",
      "agreementId": "urn:agreement:3940c7db-118d-434b-9017-7e01de13f168",
      "call

## 13. Suspend Transfer (Provider -> Consumer)
The Provider temporarily suspends data access by sending a `RpcTransferSuspensionMessage` via `/dsp/current/transfers/rpc/setup-suspension`.
The transfer process transitions from `STARTED` to `SUSPENDED`.

In [31]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "code": "SUSPEND",
    "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
    "providerPid": "urn:uuid:3b969c05-293f-4a82-9124-b2702db02fe9",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:ef4638c5-d0df-4a9f-bb5f-f5e09163dba6",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "SUSPENDED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": [
        "Suspending for demonstration purposes"
      ],
      "code": "SUSPEND"
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",

## 14. Resume Transfer (Provider -> Consumer)
In DSP 2025-1, a suspended transfer is resumed by sending another `TransferStartMessage`. Under DSP role rules, only the Provider sends start messages in PULL mode.
The Provider invokes `/dsp/current/transfers/rpc/setup-start`, transitioning the transfer process back from `SUSPENDED` to `STARTED`.

In [32]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
    "providerPid": "urn:uuid:3b969c05-293f-4a82-9124-b2702db02fe9"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:ef4638c5-d0df-4a9f-bb5f-f5e09163dba6",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
      "providerPid": "urn:uuid:3b969c05-293f-4a82-9124-b2702db02fe9",
      "agreementId": "urn:agreement:3940c7db-118d-434b-9017-7e01de13f168",
      "call

## 15. Role Validation Test: Reject Consumer-Initiated Start
DSP protocol rules enforce that only a Provider may send a `TransferStartMessage`.
If the Consumer attempts to invoke `/dsp/current/transfers/rpc/setup-start`, the request is rejected with a validation error (`only a Provider sends a TransferStartMessage`).

In [33]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
print("Status Code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)


Status Code: 400
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
  "providerPid": "urn:uuid:3b969c05-293f-4a82-9124-b2702db02fe9",
  "code": "400",
  "reason": [
    "role: only a Provider sends a TransferStartMessage",
    "state: TransferStartMessage is not allowed from state STARTED"
  ]
}


## 16. Shape Validation Test: Reject Non-RDF / Plain JSON
Eunomia RPC endpoints require valid JSON-LD messages with `@context` and `@type`.
Sending plain JSON without the Eunomia `@context` or matching `@type` is rejected by the RPC shape validator.

In [34]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
print("Status Code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)


Status Code: 400
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": "",
  "providerPid": "",
  "code": "400",
  "reason": [
    "body: expanded body declares no message type"
  ]
}


## 17. Terminate Transfer (Consumer -> Provider)
The Consumer terminates the active transfer process by sending a `RpcTransferTerminationMessage` via `/dsp/current/transfers/rpc/setup-termination`.
The transfer process moves to the terminal state `TERMINATED`.

In [35]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-termination"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferTerminationMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "STOPPED",
    "reason": ["Demonstration completed"],
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferTerminationMessage",
    "code": "STOPPED",
    "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
    "providerPid": "urn:uuid:3b969c05-293f-4a82-9124-b2702db02fe9",
    "reason": [
      "Demonstration completed"
    ]
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:b23085d6-aeaa-4c62-9b79-8062defb7b49",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "role": "consumer",
    "protocol": "dsp2025_1",
    "state": "TERMINATED",
    "stateMetadata": {
      "attribute": "BY_CONSUMER",
      "reason": [
        "Demonstration completed"
      ],
      "code": "STOPPED"
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:6cf144ca-fdb1-42f5-8d07-002185f9579d",
      "providerPid": "ur

## 18. Query Final Transfer Process State
Query the Provider's Transfer Agent Management API (`/api/v1/transfer-agent/transfer-processes`) to verify the final recorded state and audit history.

In [18]:
url = data_space_provider + "/api/v1/transfer-agent/transfer-processes"
response = requests.request("GET", url, headers=json_header)
print("Processes in Provider:")
print(json.dumps(response.json(), indent=2))


Processes in Provider:
{
  "items": [
    {
      "id": "urn:transfer-process:353c621b-3f5b-4420-aaaf-ed777380f088",
      "userId": "system",
      "userRole": "/admin",
      "visibility": "Public",
      "role": "provider",
      "protocol": "dsp2025_1",
      "state": "TERMINATED",
      "stateMetadata": {
        "attribute": "BY_CONSUMER",
        "reason": [
          "Demonstration completed"
        ],
        "code": "STOPPED"
      },
      "correlation": {
        "identifiers": {
          "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
          "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd"
        },
        "consumerPid": "urn:uuid:d69109a4-eb7d-422c-a9cb-6c6337fe8e8d",
        "providerPid": "urn:uuid:dd02dee9-8c78-46b6-96f4-8ae3342decdd",
        "agreementId": "urn:agreement:ed676de1-2364-47c1-b8af-cb019ace23b4",
        "callbackAddress": "http://127.0.0.1:1100/dsp/current",
        "peerParticipantId": "did:jwk:eyJlIjoiQVFBQiIsImt0e